<a href="https://colab.research.google.com/github/kaansoftware34/softito_calismalar_face2face/blob/main/scrape_ornegi_llm190626.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install playwright

In [ ]:
!playwright install
!playwright install-deps

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entr

In [ ]:

!cd playwright_scraper

/bin/bash: line 1: cd: playwright_scraper: No such file or directory


In [ ]:
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as p:
        # Colab'de ekran olmadığı için headless=True (varsayılan) olmalıdır
        browser = await p.chromium.launch(headless=True)
        print("Tarayıcı başarıyla başlatıldı!")

        # Tarayıcıyı kapat
        await browser.close()

# Colab asenkron çalıştırmayı doğrudan destekler
await main()

Tarayıcı başarıyla başlatıldı!


In [ ]:
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as p:
        # Tarayıcıyı başlat
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(viewport={"width": 1920, "height": 1080})

        print("arXiv (Makale veritabanı) sitesinde 'AI' araması yapılıyor...")
        # Arayüz öğeleriyle uğraşmak yerine doğrudan arama URL'sine gidiyoruz
        await page.goto("https://arxiv.org/search/?query=AI&searchtype=all")

        print("Arama sonuçları bekleniyor...")
        # Arama sonuçlarındaki başlıkların (.title) yüklenmesini bekle
        await page.wait_for_selector("p.title")

        # Sayfadaki tüm makale başlıklarını al
        titles = await page.locator("p.title").all_inner_texts()

        print("\n--- Bulunan İlk 10 AI Makalesi ---")
        for i, title in enumerate(titles[:10], 1):
            # Başlıkların başındaki gereksiz boşlukları temizleyelim
            clean_title = title.replace("\n", "").strip()
            print(f"{i}. {clean_title}")

        # Tarayıcıyı kapat
        await browser.close()
        print("\nTarayıcı kapatıldı.")

await main()

arXiv (Makale veritabanı) sitesinde 'AI' araması yapılıyor...
Arama sonuçları bekleniyor...

--- Bulunan İlk 10 AI Makalesi ---
1. Distributed Attacks in Persistent-State AI Control
2. Embodied.cpp: A Portable Inference Runtime of Embodied AI Models on Heterogeneous Robots
3. Beyond Adam: SOAP and Muon for Faster, Label-Efficient Training of Machine Learning Interatomic Potentials
4. Seek to Segment: Active Perception for Panoramic Referring Segmentation
5. Human Capital, Not Model Benchmarks, Predicts Hybrid Intelligence in Forecasting
6. Learning to Move Before Learning to Do: Task-Agnostic pretraining for VLAs
7. Probabilistic Memory for Trustworthy Edge Intelligence
8. Adoption and Ecosystem Health: A Longitudinal Analysis of Open-Source Multi-Agent Frameworks
9. MARVEL: Margin-Aware Robust von Mises-Fischer Expert Learning for Long-Tailed Out-of-Distribution Detection
10. Automated grading of Linux/bash examinations using large language models: a four-level cognitive taxonomy appr

### Adım 1: Gerekli Kütüphanelerin Kurulumu
Mistral eğitimi için (PEFT, trl, bitsandbytes) ve PDF tam metin çekme işlemleri için (arxiv, pypdf) gerekli kütüphaneleri kuruyoruz.

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes arxiv pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 145.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.8 MB/s eta 0:00:00


### Adım 2: arXiv'den Tam Metin (Full Text) Verisi Çekme
10 adet 'AI' ile ilgili makalenin PDF'lerini indirip içindeki tüm kelimeleri okuyoruz ve model eğitimi için bir JSON veri setine dönüştürüyoruz.

In [ ]:
!pip install -q arxiv pypdf requests

import arxiv
import pypdf
import json
import os
import requests

# arXiv üzerinden AI araması yapıp 10 makale alalım
client = arxiv.Client()
search = arxiv.Search(
  query = "AI",
  max_results = 10,
  sort_by = arxiv.SortCriterion.Relevance
)

dataset = []
print("Makaleler indiriliyor ve tam metinleri çıkarılıyor (Bu işlem biraz vakit alabilir)...")

for result in client.results(search):
    pdf_path = "temp_article.pdf"
    text_content = ""

    try:
        # PDF'i pdf_url üzerinden indir
        response = requests.get(result.pdf_url)
        with open(pdf_path, 'wb') as f:
            f.write(response.content)

        # PDF'in içindeki tüm sayfaları oku
        with open(pdf_path, 'rb') as f:
            reader = pypdf.PdfReader(f)
            for page in reader.pages:
                # Tam metni çıkart
                extracted = page.extract_text()
                if extracted:
                    text_content += extracted + "\n"

        print(f"Başarıyla eklendi: {result.title}")
        # Makalenin tam metnini veri setine ekle
        dataset.append({"text": f"Title: {result.title}\n\nFull Text:\n{text_content}"})
    except Exception as e:
        print(f"Hata ({result.title}): {e}")
    finally:
        # Disk dolmaması için indirilen geçici PDF'i sil
        if os.path.exists(pdf_path):
            os.remove(pdf_path)

# Tam metin veri setini JSON olarak kaydet
with open("arxiv_full_text_dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False)

print(f"\nToplam {len(dataset)} makalenin TAM METNİ 'arxiv_full_text_dataset.json' dosyasına kaydedildi!")

Makaleler indiriliyor ve tam metinleri çıkarılıyor (Bu işlem biraz vakit alabilir)...
Başarıyla eklendi: AI prediction leads people to forgo guaranteed rewards
Başarıyla eklendi: Foundations of GenIR


Başarıyla eklendi: Competing Visions of Ethical AI: A Case Study of OpenAI
Başarıyla eklendi: Towards The Ultimate Brain: Exploring Scientific Discovery with ChatGPT AI
Başarıyla eklendi: Meaningful human control: actionable properties for AI system development
Başarıyla eklendi: Need of AI in Modern Education: in the Eyes of Explainable AI (xAI)
Başarıyla eklendi: Beyond principlism: Practical strategies for ethical AI use in research practices
Başarıyla eklendi: DeBiasMe: De-biasing Human-AI Interactions with Metacognitive AIED (AI in Education) Interventions
Başarıyla eklendi: Expansive Participatory AI: Supporting Dreaming within Inequitable Institutions
Başarıyla eklendi: Exploring utilization of generative AI for research and education in data-driven materials science

Toplam 10 makalenin TAM METNİ 'arxiv_full_text_dataset.json' dosyasına kaydedildi!


### Adım 3: Mistral 7B Modelini Eğitme (Fine-Tuning)
Elde ettiğimiz bu binlerce kelimelik veriyle **Mistral-7B** modelini QLoRA (4-bit) kullanarak eğitiyoruz.
*(Not: Resmî Mistral modeli kullanmak için Hugging Face hesabınıza giriş yapmanız gerekebilir. Eğer hata alırsanız HuggingFace token'ınızı Colab'e eklemelisiniz veya 'model_id' kısmını ungated bir modelle değiştirebilirsiniz.)*

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 1. Tam metin veri setini yükle
dataset = load_dataset("json", data_files="arxiv_full_text_dataset.json", split="train")

# 2. Model ayarları (Mistral 7B)
model_id = "mistralai/Mistral-7B-v0.1"

# Colab T4 GPU belleğine (15GB) sığması için 4-bit (QLoRA) ayarları
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Mistral 7B modeli 4-bit olarak yükleniyor (Bu işlem 5-10 dakika sürebilir)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. LoRA Ayarları (Sadece belirli ağırlıkları eğiterek bellekten tasarruf ediyoruz)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)

# 4. Eğitim Ayarları (Tam metinler uzun olduğu için max_seq_length ayarlıyoruz)
training_args = TrainingArguments(
    output_dir="./mistral_finetuned_arxiv",
    per_device_train_batch_size=1,      # GPU belleğini korumak için
    gradient_accumulation_steps=4,      # Batch size'ı simüle etmek için
    warmup_steps=2,
    max_steps=20,                       # Demo amaçlı 20 adım (Gerçek eğitimde artırın)
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    optim="paged_adamw_8bit",           # 8-bit optimizer ile bellek tasarrufu
    report_to="none"
)

print("Eğitim başlıyor...")
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,                # Modele her seferinde beslenecek maksimum kelime (token) uzunluğu
    tokenizer=tokenizer,
    args=training_args,
)

# Eğitimi başlat
trainer.train()
print("✅ Eğitim tamamlandı! Yeni Mistral modeliniz './mistral_finetuned_arxiv' klasörüne kaydedildi.")

Generating train split: 0 examples [00:00, ? examples/s]

Tokenizer yükleniyor...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Eğitim başlıyor...


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'